# Clasificación en Tiempo Real con Cámara Web

In [37]:
# Importo las librerías necesarias
import cv2
import numpy as np
from tensorflow import keras

In [38]:
# Cargo el modelo entrenado
model = keras.models.load_model('91porciento.keras')
print("Modelo cargado exitosamente")
print(f"Arquitectura del modelo: {model.input_shape}")

Modelo cargado exitosamente
Arquitectura del modelo: (None, 300, 300, 3)


In [39]:
# Defino las clases
class_names = ['bird', 'monkey', 'boar', 'tiger', 'rat', 'ram', 'dog', 'horse', 'hare', 'ox', 'dragon', 'snake']  # Reemplazar con tus clases reales
num_classes = len(class_names)
print(f"Número de clases: {num_classes}")
print(f"Clases: {class_names}")

Número de clases: 12
Clases: ['bird', 'monkey', 'boar', 'tiger', 'rat', 'ram', 'dog', 'horse', 'hare', 'ox', 'dragon', 'snake']


In [40]:
def preprocess_frame(frame, target_size=(300, 300)):
    """
    Preprocesa el frame para la predicción
    """
    # Redimensionar la imagen
    resized = cv2.resize(frame, target_size)

    # Agregar dimensión del batch
    # Necesario por el formato de entrada del modelo
    batch_frame = np.expand_dims(resized, axis=0)

    return batch_frame

In [41]:
def draw_predictions(frame, predictions, class_names):
    """
    Dibuja las predicciones en el frame
    """
    height, width = frame.shape[:2]
    
    # Configuración del texto
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.7
    thickness = 2
    
    # Fondo para el texto
    overlay = frame.copy()
    cv2.rectangle(overlay, (10, 10), (400, 30 + len(class_names) * 30), (0, 0, 0), -1)
    frame = cv2.addWeighted(frame, 0.7, overlay, 0.3, 0)
    
    # Título
    cv2.putText(frame, "Predicciones:", (15, 30), font, font_scale, (255, 255, 255), thickness)
    
    # Mostrar cada predicción
    for i, (class_name, prob) in enumerate(zip(class_names, predictions[0])):
        text = f"{class_name}: {prob*100:.1f}%"
        y_position = 60 + i * 30
        
        # Color basado en la probabilidad
        if prob > 0.5:
            color = (0, 255, 0)  # Verde para alta probabilidad
        elif prob > 0.3:
            color = (0, 255, 255)  # Amarillo para probabilidad media
        else:
            color = (0, 0, 255)  # Rojo para baja probabilidad
        
        cv2.putText(frame, text, (15, y_position), font, font_scale, color, thickness)
    
    return frame

In [42]:
# Inicializar la cámara
cap = cv2.VideoCapture(0)

# Verificar si la cámara se abrió correctamente
if not cap.isOpened():
    print("Error: No se pudo abrir la cámara")
else:
    print("Cámara inicializada correctamente")
    print("Presiona 'q' para salir")

Cámara inicializada correctamente
Presiona 'q' para salir


In [43]:
# Loop principal de clasificación en tiempo real
try:
    i = 0
    while True:
        # Capturar frame
        ret, frame = cap.read()
        
        if not ret:
            print("Error: No se pudo capturar el frame")
            break
        
        # Preprocesar el frame
        processed_frame = preprocess_frame(frame)
        
        # Realizar predicción
        predictions = model.predict(processed_frame, verbose=0)
        
        # Dibujar predicciones en el frame
        frame_with_predictions = draw_predictions(frame, predictions, class_names)
        
        # Mostrar el frame
        cv2.imshow('Clasificacion en Tiempo Real', frame_with_predictions)
        
        # Salir con 'q'
        if cv2.waitKey(20) & 0xFF == ord('q'):
            break
    
except KeyboardInterrupt:
    print("\nInterrumpido por el usuario")

finally:
    # Limpiar recursos
    cap.release()
    cv2.destroyAllWindows()
    print("Recursos liberados")

Recursos liberados
